In [ ]:
# Diffusion model dependencies (TabDDPM + CoDi)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# CoDi: ChaejeongLee/CoDi (_vendor/CoDi)
%pip install -q ForestDiffusion xgboost category-encoders libzero rtdl imbalanced-learn absl-py tensorboardX

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "goggle" / "src"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "CoDi"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_codi


In [ ]:
pip install sdv

In [ ]:
pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality



# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers and sample data
data = data.replace("?", np.nan)
data = data.sample(n=1000, random_state=42).reset_index(drop=True)

# Fill missing values
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].mean())

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare features, target, and metadata
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

# Set experiment constants and seeds
N_SAMPLES = 10000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Initialize result containers
scores = {}
synthetic_datasets = {}
quality_results = []

In [ ]:
# Single run

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Train/test split without leakage

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
    stratify=processed_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[
        col for col in train_real.columns
        if col != target_col and col in label_encoders
    ],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)


In [ ]:
# CoDi

try:
    import traceback

    print("Training CoDi...")
    synthetic_codi = train_codi(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["CoDi"] = synthetic_codi.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_codi,
        metadata=train_metadata,
    )

    scores["CoDi"] = quality.get_score()

    print("CoDi:", round(scores["CoDi"], 4))

    del synthetic_codi

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("CoDi Failed:")
    traceback.print_exc()


In [ ]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None
):
    if seeds is None:
        seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            # Extract X and y from the full training DataFrame for the current seed
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]

            # Extract X and y from the full testing DataFrame for the current seed
            X_test_full = test_df.drop(columns=[label_col])
            y_test_full = test_df[label_col]

            # Determine if stratification is possible for y_train_full
            stratify_y_train_full = y_train_full if y_train_full.value_counts().min() >= 2 else None
            if stratify_y_train_full is None:
                print(
                    f"Warning: Cannot stratify training data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `train_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the first train-test split (from train_df to get actual training set for classifier)
            X_train_split, _, y_train_split, _ = train_test_split(
                X_train_full,
                y_train_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_train_full
            )

            # Determine if stratification is possible for y_test_full
            # This is usually for real data and should be fine, but included for robustness.
            stratify_y_test_full = y_test_full if y_test_full.value_counts().min() >= 2 else None
            if stratify_y_test_full is None:
                print(
                    f"Warning: Cannot stratify testing data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `test_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the second train-test split (from test_df to get actual testing set for classifier)
            _, X_test_split, _, y_test_split = train_test_split(
                X_test_full,
                y_test_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_test_full
            )

            scaler = StandardScaler().fit(X_train_split)

            X_train_s = scaler.transform(X_train_split)
            X_test_s = scaler.transform(X_test_split)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train_split)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test_split, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

        acc_mean = np.mean(accuracy_scores)
        acc_std = np.std(accuracy_scores, ddof=1)

        f1_mean = np.mean(f1_scores)
        f1_std = np.std(f1_scores, ddof=1)

        prec_mean = np.mean(precision_scores)
        prec_std = np.std(precision_scores, ddof=1)

        rec_mean = np.mean(recall_scores)
        rec_std = np.std(recall_scores, ddof=1)

        results.append({
            "Model": name,

            "Accuracy Mean": acc_mean,
            "Accuracy Std": acc_std,
            "F1 Mean": f1_mean,
            "F1 Std": f1_std,
            "Precision Mean": prec_mean,
            "Precision Std": prec_std,
            "Recall Mean": rec_mean,
            "Recall Std": rec_std,

            "Accuracy \u00b1 SD": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 \u00b1 SD": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision \u00b1 SD": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall \u00b1 SD": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}",

            "Accuracy (Mean\u00b1Std)": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 (Mean\u00b1Std)": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision (Mean\u00b1Std)": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall (Mean\u00b1Std)": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )

In [ ]:
# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers
data = data.replace("?", np.nan)

# Clean income target into two classes only
data[target_col] = (
    data[target_col]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.strip()
)

print(data[target_col].value_counts())

# Take 1000 real samples
data = data.sample(n=1000, random_state=42).reset_index(drop=True)

# Fill missing values
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].mean())

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare processed dataset
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

print("\nEncoded income classes:")
print(processed_data[target_col].value_counts())
print(label_encoders[target_col].classes_)

In [ ]:
# TRTR evaluation for Adult income dataset

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

X = processed_data.drop(columns=[target_col])
y = processed_data[target_col]

print("Target column:", target_col)
print("Target classes:")
print(y.value_counts())

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores, ddof=1)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores, ddof=1)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores, ddof=1)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores, ddof=1)

    trtr_results.append({
        "Model": model_name,

        "Accuracy Mean_TRTR": acc_mean,
        "Accuracy Std_TRTR": acc_std,
        "F1 Mean_TRTR": f1_mean,
        "F1 Std_TRTR": f1_std,
        "Precision Mean_TRTR": prec_mean,
        "Precision Std_TRTR": prec_std,
        "Recall Mean_TRTR": rec_mean,
        "Recall Std_TRTR": rec_std,

        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results).sort_values(
    by="Accuracy Mean_TRTR",
    ascending=False
)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)

In [ ]:
import pandas as pd

label_col = target_col   # Adult dataset target column: income

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "CoDi",
    "TabDDPM"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

real_data = processed_data.copy()

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=real_data,
    test_df=real_data,
    label_col=label_col,
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy \u00b1 SD",
            "F1 \u00b1 SD",
            "Precision \u00b1 SD",
            "Recall \u00b1 SD"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"{synth_name} not found in synthetic_datasets. Skipping.")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name].copy()

    # Ensure the target column in the synthetic data is integer-encoded (0 or 1)
    # consistent with the real_data. The `label_encoders` dictionary from previous cells
    # contains the LabelEncoder fitted on the original string labels.
    if label_col in synthetic_train_df.columns:
        if pd.api.types.is_object_dtype(synthetic_train_df[label_col]):
            # If the column contains string labels, transform them to integers
            synthetic_train_df[label_col] = label_encoders[label_col].transform(synthetic_train_df[label_col])
        elif not pd.api.types.is_integer_dtype(synthetic_train_df[label_col]):
            # If it's numeric but not integer (e.g., float), convert to integer
            synthetic_train_df[label_col] = synthetic_train_df[label_col].astype(int)

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=real_data,
        label_col=label_col,
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy \u00b1 SD",
                "F1 \u00b1 SD",
                "Precision \u00b1 SD",
                "Recall \u00b1 SD"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=('_TRTR', '_TSTR')
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy \u00b1 SD_TRTR",
                "Accuracy \u00b1 SD_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


In [ ]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")